# SolarSentinel AI - 01: Dataset Analysis & Integrity Audit

This notebook performs a comprehensive exploratory data analysis (EDA) and data integrity audit on the Solar Panel Defect Dataset.

### Core Objectives:
1. Audit total image count, file formats, and channel configurations.
2. Analyze class distribution and quantify class imbalance.
3. Detect byte-level duplicate images via SHA-256 hash collision checking.
4. Verify the deterministic 70/15/15 stratified train/val/test split and prove zero partition leakage.

In [ ]:
import os
import sys
import json
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ml.data.dataset_audit import audit_dataset
from ml.data.duplicate_detection import detect_duplicates
from ml.data.create_split import verify_existing_split

## 1. Dataset Dimensions & Image Integrity Audit

In [ ]:
audit_report = audit_dataset()
print("Total Images Scanned:", audit_report["total_images"])
print("Corrupted Files:", audit_report["corrupted_count"])
print("Color Channels:", audit_report["channel_modes"])
print("\nTop 5 Resolutions:")
for res, count in audit_report["top_resolutions"]:
    print(f"  {res}: {count} images")

## 2. Class Distribution & Imbalance Analysis

In [ ]:
classes = list(audit_report["class_counts"].keys())
counts = list(audit_report["class_counts"].values())

plt.figure(figsize=(10, 5), dpi=150)
bars = plt.bar(classes, counts, color="#2563EB", edgecolor="#1E3A8A", alpha=0.85)
plt.title("Solar Panel Fault Dataset - Class Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Defect Category", fontsize=10, fontweight="bold")
plt.ylabel("Number of Samples", fontsize=10, fontweight="bold")
plt.xticks(rotation=30, ha="right")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval}", ha="center", va="bottom", fontweight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Majority Class: {classes[counts.index(max(counts))]} ({max(counts)} samples)")
print(f"Minority Class: {classes[counts.index(min(counts))]} ({min(counts)} samples)")
print(f"Imbalance Ratio: {max(counts)/min(counts):.2f}:1")

## 3. Duplicate Detection & Split Leakage Verification

Data leakage between train, validation, and test partitions invalidates reported scientific metrics.
We compute SHA-256 hashes for all 885 images and assert that identical images are strictly confined to a single split.

In [ ]:
dup_report = detect_duplicates()
print("=" * 50)
print(f"Unique Hashes:        {dup_report['unique_images_by_hash']}")
print(f"Duplicate Groups:     {dup_report['duplicate_hash_groups']}")
print(f"Redundant Images:     {dup_report['total_duplicate_redundant_files']}")
print(f"Intra-Class Dups:     {dup_report['intra_class_duplicate_groups']}")
print(f"Cross-Class Dups:     {dup_report['cross_class_duplicate_groups']}")
print(f"Split Leakage Detected: {dup_report['split_leakage_detected']} ({dup_report['split_leakage_count']} instances)")
print("=" * 50)
assert not dup_report['split_leakage_detected'], "CRITICAL: Partition leakage detected!"

## 4. Stratified 70/15/15 Partition Verification

In [ ]:
split_verif = verify_existing_split()
print("Split Counts:", split_verif["counts"])
df_splits = pd.DataFrame(split_verif["distributions"])
df_splits["Total"] = df_splits.sum(axis=1)
df_splits